Historical plotting notebook. Start Jupyter from the repository root; use the experiment READMEs and command-line runners to generate inputs first. Cells retain exploratory plotting variants; Run All is not the reproduction entry point.

In [ ]:
from pathlib import Path
import os, sys
while not Path("pyproject.toml").exists():
    if Path.cwd() == Path.cwd().parent: raise RuntimeError("Start Jupyter inside this repository")
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import glob
import re

In [ ]:


# Create output directory for experiment 5
output_dir = "exp5_plots"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Set global plotting style parameters for consistency
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
plt.rcParams['font.size'] = 14
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 14


In [ ]:


# Helper function to plot GI distributions
def plot_gi_distribution(model_folders, dataset, model_names, ylim_range=(70, 102), legend_position='lower right'):
    """
    Plot GI distribution boxplots for specified models and dataset.
    
    Parameters:
    - model_folders: List of folder paths containing model results
    - dataset: Name of the dataset (for title and filename)
    - model_names: List of model names to display
    - ylim_range: Tuple with (min, max) for y-axis limits
    - legend_position: Position of the legend (e.g., 'lower right', 'upper left')
    """
    # Data structures to store our results
    all_models_correct_data = []
    all_models_misclassified_data = []
    positions = []

    # Check if any model has misclassified data
    has_any_misclassified = False
    for folder in model_folders:
        misclassified_files = glob.glob(os.path.join(folder, "misclassified_from_class_*.csv"))
        if misclassified_files:
            has_any_misclassified = True
            break

    # Create compact figure with minimal margins
    num_models = len(model_names)
    width_per_model = 1.2 if has_any_misclassified else 1.0
    fig_width = max(6, num_models * width_per_model)
    fig_height = 3  # shorter height
    plt.figure(figsize=(fig_width, fig_height), facecolor='white')

    plt.subplots_adjust(left=0.1, right=0.95, top=0.9, bottom=0.2)

    # Colors for correct and misclassified
    colors = ['#A8D5BA', '#F4A7A7']  # mild green for correct, mild red for misclassified

    # Position counter for plotting
    position_counter = 0

    # Lists to store box plots for the legend
    box_correct = None
    box_misclassified = None

    # Loop through each model's results folder
    for i, (folder, model_name) in enumerate(zip(model_folders, model_names)):
        print(f"\nProcessing {model_name} from folder: {folder}")
        
        # Get all CSV files for correct classifications
        correct_files = glob.glob(os.path.join(folder, "from_train_to_test_GIs_class_*.csv"))
        # Get all CSV files for misclassifications
        misclassified_files = glob.glob(os.path.join(folder, "misclassified_from_class_*.csv"))
        
        print(f"Found {len(correct_files)} correct classification files")
        print(f"Found {len(misclassified_files)} misclassification files")
        
        # Process correct classification files
        correct_data = []
        for file in correct_files:
            try:
                df = pd.read_csv(file)
                if df.empty:
                    continue
                    
                # Extract percent_gi_above_1 if it exists
                if "percent_gi_above_1" in df.columns:
                    correct_data.append(df["percent_gi_above_1"].iloc[0])
                # If the file has a GI in the first column with no header, use that
                elif df.shape[1] >= 1 and df.columns[0].startswith("Unnamed"):
                    values = df.iloc[:, 0].dropna().astype(float).tolist()
                    correct_data.extend(values)
                else:
                    # Try to extract numeric data from the first column
                    values = df.iloc[:, 0].dropna()
                    if values.apply(lambda x: isinstance(x, (int, float))).all():
                        correct_data.extend(values.astype(float).tolist())
                    
            except Exception as e:
                print(f"Error reading {os.path.basename(file)}: {e}")
        
        # Process misclassification files
        misclassified_data = []
        if has_any_misclassified:  # Only try to process misclassified data if any model has it
            for file in misclassified_files:
                try:
                    df = pd.read_csv(file)
                    if df.empty:
                        continue
                        
                    # Extract percent_gi_above_1 if it exists
                    if "percent_gi_above_1" in df.columns:
                        misclassified_data.append(df["percent_gi_above_1"].iloc[0])
                    # If the file has a GI in the first column with no header, use that
                    elif df.shape[1] >= 1 and df.columns[0].startswith("Unnamed"):
                        values = df.iloc[:, 0].dropna().astype(float).tolist()
                        misclassified_data.extend(values)
                    else:
                        # Try to extract numeric data from the first column
                        values = df.iloc[:, 0].dropna()
                        if values.apply(lambda x: isinstance(x, (int, float))).all():
                            misclassified_data.extend(values.astype(float).tolist())
                        
                except Exception as e:
                    print(f"Error reading {os.path.basename(file)}: {e}")
        
        # Skip this model if we didn't find any data
        if not correct_data and not misclassified_data:
            print(f"No valid data found for {model_name}, skipping...")
            continue
        
        # Store the data
        all_models_correct_data.append(correct_data)
        all_models_misclassified_data.append(misclassified_data)
        
        # Print some stats
        print(f"Model: {model_name}")
        print(f"Correct samples: {len(correct_data)}")
        print(f"Misclassified samples: {len(misclassified_data)}")
        if correct_data:
            print(f"Correct GI range: {min(correct_data)} to {max(correct_data)}")
        if misclassified_data:
            print(f"Misclassified GI range: {min(misclassified_data)} to {max(misclassified_data)}")

        # Calculate positions for the box plots
        if has_any_misclassified:
            # If we have misclassified data for any model, create positions for both correct and misclassified
            pos_correct = position_counter
            pos_misclassified = position_counter + 0.4
            positions.append((pos_correct, pos_misclassified))
            position_counter += 1.2
        else:
            # If no misclassified data, just use a single position per model
            pos = position_counter
            positions.append(pos)
            position_counter += 1.0

    # Now plot all the data
    for i, (model_name, correct_data, misclassified_data) in enumerate(zip(model_names, all_models_correct_data, all_models_misclassified_data)):
        if has_any_misclassified:
            # Plot pairs of box plots (correct and misclassified)
            pos_correct, pos_misclassified = positions[i]
            
            # Plot correct data
            if correct_data:
                box1 = plt.boxplot([correct_data], positions=[pos_correct], 
                        patch_artist=True, widths=0.35, showfliers=False)
                # Color the boxes
                for patch in box1['boxes']:
                    patch.set_facecolor(colors[0])
                
                # Save the box objects for the legend
                if box_correct is None:
                    box_correct = box1
            
            # Plot misclassified data if available
            if misclassified_data:
                box2 = plt.boxplot([misclassified_data], positions=[pos_misclassified], 
                        patch_artist=True, widths=0.35, showfliers=False)
                # Color the boxes
                for patch in box2['boxes']:
                    patch.set_facecolor(colors[1])
                    
                # Save the box objects for the legend
                if box_misclassified is None:
                    box_misclassified = box2
        else:
            # Plot single box plots (correct only)
            pos = positions[i]
            
            # Plot correct data
            if correct_data:
                box = plt.boxplot([correct_data], positions=[pos], 
                        patch_artist=True, widths=0.6, showfliers=False)
                # Color the boxes
                for patch in box['boxes']:
                    patch.set_facecolor(colors[0])
                
                # Save the box objects for the legend
                if box_correct is None:
                    box_correct = box

    # Set x-tick positions and labels
    if positions:
        if has_any_misclassified:
            # For datasets with both correct and misclassified
            model_positions = [sum(pos)/2 for pos in positions]
            plt.xticks(model_positions, model_names)
        else:
            # For datasets with only correct classifications
            plt.xticks(positions, model_names)
        
        # Add line at GI=1 (threshold for generalization)
        plt.axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
        
        # Set labels and title with reduced font size for compactness
        plt.ylabel('Percentage of GI > 1', fontsize=12)
        plt.title(f'{dataset} - GI Distribution Across Models', fontsize=14)
        
        # Create custom legend based on available data
        legend_elements = []
        legend_labels = []
        
        if box_correct:
            legend_elements.append(box_correct["boxes"][0])
            legend_labels.append('Correctly Classified')
        
        if box_misclassified:
            legend_elements.append(box_misclassified["boxes"][0])
            legend_labels.append('Misclassified')
        
        if legend_elements:
            plt.legend(legend_elements, legend_labels, loc=legend_position, 
                      frameon=True, framealpha=0.8, fontsize=10)
        
        # Enhance appearance
        plt.grid(axis='y', linestyle='--', alpha=0.3)
        
        # Add y-axis limits for cleaner appearance
        plt.ylim(ylim_range)  # Use the custom y-limits
        
        # Adjust layout
        plt.tight_layout()
        
        # Save figure in both formats
        plt.savefig(os.path.join(output_dir, f'{dataset}_model_comparison_GIs.png'), 
                   dpi=300, bbox_inches='tight', facecolor='white')
        plt.savefig(os.path.join(output_dir, f'{dataset}_model_comparison_GIs.pdf'), 
                   bbox_inches='tight', facecolor='white')
        plt.show()
    else:
        print("No data found for plotting. Check your file paths and CSV format.")

In [ ]:
# MNIST Folders
mnist_model_folders = [
    "experiments/results/test_to_train_gis/mnist_original_mlp/GI_experiment_outputs_plainnn",
    "experiments/results/test_to_train_gis/mnist_original_cnn/GI_experiment_outputs_cnn",
    "experiments/results/test_to_train_gis/mnist_original_vit/GI_experiment_outputs_vit"
]
mnist_model_names = ['MLP', 'CNN', 'ViT']

# Plot MNIST data with custom y-limits
plot_gi_distribution(
    model_folders=mnist_model_folders,
    dataset='MNIST',
    model_names=mnist_model_names,
    ylim_range=(70, 102),  # Typical range for MNIST
    legend_position='lower right'
)

In [ ]:
# CIFAR10 folders
cifar10_model_folders = [
    "experiments/results/test_to_train_gis/cifar_10_resnet_20/GI_cifar10_real_resnet20_experiment_outputs",
    "experiments/results/test_to_train_gis/cifar10_vgg16_bn/GI_cifar10_real_vgg16_bn_experiment_outputs"
]
cifar10_model_names = ['ResNet20', 'VGG16']

# Plot CIFAR10 data with custom y-limits
plot_gi_distribution(
    model_folders=cifar10_model_folders,
    dataset='CIFAR10',
    model_names=cifar10_model_names,
    ylim_range=(0, 100),  # Adjusted range for CIFAR10
    legend_position='lower right'
)

In [ ]:
# Imagenette folders
imagenette_model_folders = [
    "experiments/results/test_to_train_gis/imagenette_vgg16_random/GI_imagenette_real_VGG16_experiment_outputs",
    "experiments/results/test_to_train_gis/imagenette_resnet50/GI_imagenette_real_ResNet50_experiment_outputs",
    "experiments/results/test_to_train_gis/imagenette_deit_small/GI_imagenette_real_DeiT-Small_experiment_outputs",
    "experiments/results/test_to_train_gis/imagenette_vit_base/GI_imagenette_real_ViT-Base_experiment_outputs"
]
imagenette_model_names = ['VGG16', 'ResNet50', 'DeiT-S', 'ViT-B']

# Plot Imagenette data with custom y-limits
plot_gi_distribution(
    model_folders=imagenette_model_folders,
    dataset='Imagenette',
    model_names=imagenette_model_names,
    ylim_range=(0, 105),  # Adjusted range for Imagenette
    legend_position='lower right'
)

In [ ]:
# MNIST Random
mnist_random_model_folders = [
    "data/MNIST-RANDOM/GI_random_experiment_outputs",
    "data/MNIST-RANDOM-100/GI_random_experiment_outputs"
]
mnist_random_model_names = ['RANDOM-10', 'RANDOM-100']

# Plot MNIST-RANDOM data with custom y-limits
plot_gi_distribution(
    model_folders=mnist_random_model_folders,
    dataset='MNIST-RANDOM',
    model_names=mnist_random_model_names,
    ylim_range=(0, 80),  # Adjusted range for random labels
    legend_position='upper left'
)

In [ ]:
def create_combined_model_comparison_plot(
    mnist_data, cifar10_data, imagenette_data,
    ylim_ranges=[(70, 102), (0, 100), (0, 105)],
    legend_positions=['lower right', 'lower right', 'lower right']
):
    """
    Create a combined plot with three datasets (MNIST, CIFAR10, Imagenette) arranged horizontally.
    
    Parameters:
    - mnist_data: Tuple of (model_folders, model_names) for MNIST
    - cifar10_data: Tuple of (model_folders, model_names) for CIFAR10
    - imagenette_data: Tuple of (model_folders, model_names) for Imagenette
    - ylim_ranges: List of (min, max) tuples for y-axis limits for each dataset
    - legend_positions: List of legend positions for each dataset
    """
    # Create a figure with three horizontally arranged subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor='white')
    
    # Data for all datasets
    datasets = ["MNIST", "CIFAR10", "Imagenette"]
    all_data = [mnist_data, cifar10_data, imagenette_data]
    
    # Process each dataset in its own subplot
    for idx, ((model_folders, model_names), dataset, ylim_range, legend_pos) in enumerate(
            zip(all_data, datasets, ylim_ranges, legend_positions)):
        
        # Select the current subplot
        ax = axes[idx]
        plt.sca(ax)
        
        # Data structures to store our results
        all_models_correct_data = []
        all_models_misclassified_data = []
        positions = []

        # Check if any model has misclassified data
        has_any_misclassified = False
        for folder in model_folders:
            misclassified_files = glob.glob(os.path.join(folder, "misclassified_from_class_*.csv"))
            if misclassified_files:
                has_any_misclassified = True
                break

        # Colors for correct and misclassified
        colors = ['#A8D5BA', '#F4A7A7']  # mild green for correct, mild red for misclassified

        # Position counter for plotting
        position_counter = 0

        # Lists to store box plots for the legend
        box_correct = None
        box_misclassified = None

        # Loop through each model's results folder
        for i, (folder, model_name) in enumerate(zip(model_folders, model_names)):
            print(f"\nProcessing {dataset} - {model_name} from folder: {folder}")
            
            # Get all CSV files for correct classifications
            correct_files = glob.glob(os.path.join(folder, "from_train_to_test_GIs_class_*.csv"))
            # Get all CSV files for misclassifications
            misclassified_files = glob.glob(os.path.join(folder, "misclassified_from_class_*.csv"))
            
            print(f"Found {len(correct_files)} correct classification files")
            print(f"Found {len(misclassified_files)} misclassification files")
            
            # Process correct classification files
            correct_data = []
            for file in correct_files:
                try:
                    df = pd.read_csv(file)
                    if df.empty:
                        continue
                        
                    # Extract percent_gi_above_1 if it exists
                    if "percent_gi_above_1" in df.columns:
                        correct_data.append(df["percent_gi_above_1"].iloc[0])
                    # If the file has a GI in the first column with no header, use that
                    elif df.shape[1] >= 1 and df.columns[0].startswith("Unnamed"):
                        values = df.iloc[:, 0].dropna().astype(float).tolist()
                        correct_data.extend(values)
                    else:
                        # Try to extract numeric data from the first column
                        values = df.iloc[:, 0].dropna()
                        if values.apply(lambda x: isinstance(x, (int, float))).all():
                            correct_data.extend(values.astype(float).tolist())
                        
                except Exception as e:
                    print(f"Error reading {os.path.basename(file)}: {e}")
            
            # Process misclassification files
            misclassified_data = []
            if has_any_misclassified:  # Only try to process misclassified data if any model has it
                for file in misclassified_files:
                    try:
                        df = pd.read_csv(file)
                        if df.empty:
                            continue
                            
                        # Extract percent_gi_above_1 if it exists
                        if "percent_gi_above_1" in df.columns:
                            misclassified_data.append(df["percent_gi_above_1"].iloc[0])
                        # If the file has a GI in the first column with no header, use that
                        elif df.shape[1] >= 1 and df.columns[0].startswith("Unnamed"):
                            values = df.iloc[:, 0].dropna().astype(float).tolist()
                            misclassified_data.extend(values)
                        else:
                            # Try to extract numeric data from the first column
                            values = df.iloc[:, 0].dropna()
                            if values.apply(lambda x: isinstance(x, (int, float))).all():
                                misclassified_data.extend(values.astype(float).tolist())
                            
                    except Exception as e:
                        print(f"Error reading {os.path.basename(file)}: {e}")
            
            # Skip this model if we didn't find any data
            if not correct_data and not misclassified_data:
                print(f"No valid data found for {model_name}, skipping...")
                continue
            
            # Store the data
            all_models_correct_data.append(correct_data)
            all_models_misclassified_data.append(misclassified_data)
            
            # Print some stats
            print(f"Model: {model_name}")
            print(f"Correct samples: {len(correct_data)}")
            print(f"Misclassified samples: {len(misclassified_data)}")
            if correct_data:
                print(f"Correct GI range: {min(correct_data)} to {max(correct_data)}")
            if misclassified_data:
                print(f"Misclassified GI range: {min(misclassified_data)} to {max(misclassified_data)}")

            # Calculate positions for the box plots
            if has_any_misclassified:
                # If we have misclassified data for any model, create positions for both correct and misclassified
                pos_correct = position_counter
                pos_misclassified = position_counter + 0.4
                positions.append((pos_correct, pos_misclassified))
                position_counter += 1.2
            else:
                # If no misclassified data, just use a single position per model
                pos = position_counter
                positions.append(pos)
                position_counter += 1.0

        # Now plot all the data for this dataset
        for i, (model_name, correct_data, misclassified_data) in enumerate(zip(model_names, all_models_correct_data, all_models_misclassified_data)):
            if has_any_misclassified:
                # Plot pairs of box plots (correct and misclassified)
                pos_correct, pos_misclassified = positions[i]
                
                # Plot correct data
                if correct_data:
                    box1 = ax.boxplot([correct_data], positions=[pos_correct], 
                            patch_artist=True, widths=0.35, showfliers=False)
                    # Color the boxes
                    for patch in box1['boxes']:
                        patch.set_facecolor(colors[0])
                    
                    # Save the box objects for the legend
                    if box_correct is None:
                        box_correct = box1
                
                # Plot misclassified data if available
                if misclassified_data:
                    box2 = ax.boxplot([misclassified_data], positions=[pos_misclassified], 
                            patch_artist=True, widths=0.35, showfliers=False)
                    # Color the boxes
                    for patch in box2['boxes']:
                        patch.set_facecolor(colors[1])
                        
                    # Save the box objects for the legend
                    if box_misclassified is None:
                        box_misclassified = box2
            else:
                # Plot single box plots (correct only)
                pos = positions[i]
                
                # Plot correct data
                if correct_data:
                    box = ax.boxplot([correct_data], positions=[pos], 
                            patch_artist=True, widths=0.6, showfliers=False)
                    # Color the boxes
                    for patch in box['boxes']:
                        patch.set_facecolor(colors[0])
                    
                    # Save the box objects for the legend
                    if box_correct is None:
                        box_correct = box

        # Set x-tick positions and labels
        if positions:
            if has_any_misclassified:
                # For datasets with both correct and misclassified
                model_positions = [sum(pos)/2 for pos in positions]
                ax.set_xticks(model_positions)
                ax.set_xticklabels(model_names)
            else:
                # For datasets with only correct classifications
                ax.set_xticks(positions)
                ax.set_xticklabels(model_names)
            
            # Add line at GI=1 (threshold for generalization)
            ax.axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
            
            # Set labels for the current subplot
            if idx == 0:  # Only put y-label on the leftmost subplot
                ax.set_ylabel('Percentage of GI > 1', fontsize=12)
            ax.set_title(f'{dataset}', fontsize=14)
            
            # Create custom legend based on available data
            legend_elements = []
            legend_labels = []
            
            if box_correct:
                legend_elements.append(box_correct["boxes"][0])
                legend_labels.append('Correctly Classified')
            
            if box_misclassified:
                legend_elements.append(box_misclassified["boxes"][0])
                legend_labels.append('Misclassified')
            
            if legend_elements:
                ax.legend(legend_elements, legend_labels, loc=legend_pos, 
                        frameon=True, framealpha=0.8, fontsize=10)
            
            # Enhance appearance
            ax.grid(axis='y', linestyle='--', alpha=0.3)
            
            # Add y-axis limits for cleaner appearance
            ax.set_ylim(ylim_range)  # Use the custom y-limits
        else:
            print(f"No data found for plotting {dataset}. Check your file paths and CSV format.")
    
    # Add a single overall title
    fig.suptitle('GI Distribution Across Datasets and Models', fontsize=16, y=1.02)
    
    # Adjust layout for the entire figure
    plt.tight_layout()
    
    # Save the combined figure
    plt.savefig(os.path.join(output_dir, 'combined_model_comparison_GIs.png'), 
               dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(os.path.join(output_dir, 'combined_model_comparison_GIs.pdf'), 
               bbox_inches='tight', facecolor='white')
    
    plt.show()
    print("Combined figure saved to output directory.")

# Example usage:
# create_combined_model_comparison_plot(
#     mnist_data=(mnist_model_folders, mnist_model_names),
#     cifar10_data=(cifar10_model_folders, cifar10_model_names),
#     imagenette_data=(imagenette_model_folders, imagenette_model_names)
# )

In [ ]:
# Example usage:
create_combined_model_comparison_plot(
    mnist_data=(mnist_model_folders, mnist_model_names),
    cifar10_data=(cifar10_model_folders, cifar10_model_names),
    imagenette_data=(imagenette_model_folders, imagenette_model_names)
)

# LT1 and LT2

In [ ]:
from pathlib import Path


In [ ]:
# Set global plotting style parameters for consistency
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

In [ ]:
def compute_misclassification_stats_by_class_frequency(root_folder, distribution_type, prefix=""):
    """
    Compute misclassification statistics by class frequency for long-tailed distributions.
    
    Parameters:
    - root_folder: Path to the folder containing CSV files
    - distribution_type: Type of distribution ('lt1' or 'lt2')
    - prefix: Optional prefix for output file names
    """
    root = Path(root_folder)
    
    # Define the class distributions based on the specified type
    if distribution_type.lower() == "lt1":
        train_distribution = {0: 4717, 1: 2866, 2: 2370, 3: 2015, 4: 1806, 
                             5: 1590, 6: 1346, 7: 1034, 8: 886, 9: 730}
    elif distribution_type.lower() == "lt2":
        train_distribution = {0: 4716, 1: 2504, 2: 2026, 3: 1464, 4: 1096, 
                             5: 761, 6: 383, 7: 218, 8: 93, 9: 60}
    else:
        print(f"Unknown distribution type: {distribution_type}")
        return
    
    # Get all values and calculate actual percentiles
    counts = list(train_distribution.values())
    
    # Calculate 33rd and 66th percentiles
    q1 = np.percentile(counts, 33.33)
    q2 = np.percentile(counts, 66.67)
    
    print(f"Distribution type: {distribution_type}")
    print(f"33rd percentile (q1): {q1}")
    print(f"66th percentile (q2): {q2}")
    
    # Map classes to frequency categories based on these percentiles
    frequency_map = {}
    for cls, count in train_distribution.items():
        if count >= q2:
            frequency_map[cls] = "high"
        elif count >= q1:
            frequency_map[cls] = "medium"
        else:
            frequency_map[cls] = "low"
    
    # Print the categorization for reference
    high_classes = [cls for cls, freq in frequency_map.items() if freq == "high"]
    medium_classes = [cls for cls, freq in frequency_map.items() if freq == "medium"]
    low_classes = [cls for cls, freq in frequency_map.items() if freq == "low"]
    
    print(f"High frequency classes: {sorted(high_classes)}")
    print(f"Medium frequency classes: {sorted(medium_classes)}")
    print(f"Low frequency classes: {sorted(low_classes)}")
    
    # Define order for categories - correct logical order
    freq_order = ['high', 'medium', 'low']
    
    # Container for all misclassified data
    all_misclassified_df = []
    
    # Process all misclassified CSV files
    for file in root.glob("misclassified_*.csv"):
        try:
            df = pd.read_csv(file)
            if "percent_gi_above_1" not in df.columns or "predicted_class" not in df.columns or "true_label" not in df.columns:
                print(f"Skipping {file.name}: Missing required columns")
                continue
            
            # Add the file to our collection
            all_misclassified_df.append(df)
            
        except Exception as e:
            print(f"Failed to process {file.name}: {e}")
    
    # Combine all data
    if not all_misclassified_df:
        print("No valid misclassification data found.")
        return
        
    combined_df = pd.concat(all_misclassified_df, ignore_index=True)
    
    # Convert class columns to integers if they're strings
    for col in ['true_label', 'predicted_class']:
        if combined_df[col].dtype == 'object':
            combined_df[col] = combined_df[col].astype(int)
    
    # Add frequency category columns
    combined_df['true_freq'] = combined_df['true_label'].map(lambda x: frequency_map.get(x, "unknown"))
    combined_df['pred_freq'] = combined_df['predicted_class'].map(lambda x: frequency_map.get(x, "unknown"))
    
    return combined_df, freq_order, root, prefix, distribution_type

In [ ]:
def create_misclassification_heatmap(combined_df, freq_order, root, prefix, distribution_type):
    """Create a heatmap of misclassifications by frequency category."""
    # Create a heatmap of misclassification by frequency category
    misclass_by_freq = pd.crosstab(
        combined_df['true_freq'], 
        combined_df['pred_freq'], 
        normalize='index'
    ) * 100
    
    # Reorder rows and columns to ensure high -> medium -> low ordering
    misclass_by_freq = misclass_by_freq.reindex(freq_order, axis=0)
    misclass_by_freq = misclass_by_freq.reindex(freq_order, axis=1)
    
    plt.figure(figsize=(6, 3), facecolor='white')
    sns.heatmap(misclass_by_freq, annot=True, fmt='.1f', cmap='YlGnBu')
    plt.xlabel('Predicted Class Frequency', fontsize=14)
    plt.ylabel('True Class Frequency', fontsize=14)
    plt.title(f'Misclassification Distribution (%) - {distribution_type.upper()}', fontsize=16)
    plt.tight_layout()
    
    # Save to the output directory
    output_path = os.path.join(output_dir, f"{prefix}{distribution_type}_misclass_by_freq.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    
    # Also save PDF
    output_path_pdf = os.path.join(output_dir, f"{prefix}{distribution_type}_misclass_by_freq.pdf")
    plt.savefig(output_path_pdf, bbox_inches='tight', facecolor='white')
    
    plt.close()
    print(f"Saved misclassification heatmap to {output_path}")
    
    return misclass_by_freq

In [ ]:
def create_gi_coverage_heatmap(combined_df, freq_order, root, prefix, distribution_type):
    """Create a heatmap of GI coverage by frequency category."""
    # Analyze GI coverage by frequency category
    gi_by_freq = combined_df.groupby(['true_freq', 'pred_freq'])['percent_gi_above_1'].agg(
        ['count', 'mean', 'std', 'min', 'max', 'median']
    ).reset_index()
    
    # Save detailed stats to CSV
    stats_path = os.path.join(output_dir, f"{prefix}{distribution_type}_gi_by_freq_stats.csv")
    gi_by_freq.to_csv(stats_path, index=False)
    print(f"Saved GI stats to {stats_path}")
    
    # Create a heatmap of mean GI coverage by frequency category
    gi_pivot = gi_by_freq.pivot(index='true_freq', columns='pred_freq', values='mean')
    
    # Reorder rows and columns to ensure high -> medium -> low ordering
    gi_pivot = gi_pivot.reindex(freq_order, axis=0)
    gi_pivot = gi_pivot.reindex(freq_order, axis=1)
    
    plt.figure(figsize=(6, 3), facecolor='white')
    sns.heatmap(gi_pivot, annot=True, fmt='.1f', cmap='PuBuGn')
    plt.xlabel('Predicted Class Frequency', fontsize=14)
    plt.ylabel('True Class Frequency', fontsize=14)
    plt.title(f'Average GI Coverage (% > 1) - {distribution_type.upper()}', fontsize=16)
    plt.tight_layout()
    
    # Save to the output directory
    output_path = os.path.join(output_dir, f"{prefix}{distribution_type}_gi_coverage_by_freq.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    
    # Also save PDF
    output_path_pdf = os.path.join(output_dir, f"{prefix}{distribution_type}_gi_coverage_by_freq.pdf")
    plt.savefig(output_path_pdf, bbox_inches='tight', facecolor='white')
    
    plt.close()
    print(f"Saved GI coverage heatmap to {output_path}")
    
    return gi_pivot, gi_by_freq

In [ ]:
def create_gi_bar_chart(gi_by_freq, freq_order, root, prefix, distribution_type):
    """Create a bar chart comparing GI coverage for different frequency categories."""
    # Convert categorical values to ensure right order for plotting
    gi_by_freq_sorted = gi_by_freq.copy()
    gi_by_freq_sorted['true_freq'] = pd.Categorical(gi_by_freq_sorted['true_freq'], categories=freq_order, ordered=True)
    gi_by_freq_sorted['pred_freq'] = pd.Categorical(gi_by_freq_sorted['pred_freq'], categories=freq_order, ordered=True)
    gi_by_freq_sorted = gi_by_freq_sorted.sort_values(['true_freq', 'pred_freq'])
    
    # Create x-positions for grouped bars
    x = np.arange(len(freq_order))
    width = 0.25
    
    # Create grouped bar chart
    fig, ax = plt.subplots(figsize=(12, 8), facecolor='white')
    
    # Filter data for each true_freq and plot
    for i, true_freq in enumerate(freq_order):
        true_freq_data = gi_by_freq_sorted[gi_by_freq_sorted['true_freq'] == true_freq]
        positions = x + (i - 1) * width
        bars = ax.bar(positions, true_freq_data['mean'], width, 
                     label=f'True: {true_freq}', 
                     yerr=true_freq_data['std'],
                     capsize=5)
        
        # Add count labels above bars
        for j, bar in enumerate(bars):
            if j < len(true_freq_data):
                count = true_freq_data.iloc[j]['count']
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + 2,
                        f'n={count}', ha='center', va='bottom', size=9)
    
    ax.set_ylabel('Mean GI Coverage (% > 1)', fontsize=14)
    ax.set_title(f'GI Coverage by Class Frequency - {distribution_type.upper()}', fontsize=16)
    ax.set_xticks(x)
    ax.set_xticklabels(freq_order)
    ax.set_xlabel('Predicted Class Frequency', fontsize=14)
    ax.legend(title='True Class Frequency')
    ax.grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    output_path = os.path.join(output_dir, f"{prefix}{distribution_type}_gi_freq_comparison.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved GI frequency comparison chart to {output_path}")

In [ ]:
def create_migration_pattern(combined_df, freq_order, root, prefix, distribution_type):
    """Create a flow diagram showing migration patterns between class frequencies."""
    # Create a flow diagram showing migration patterns
    flow_counts = pd.crosstab(combined_df['true_freq'], combined_df['pred_freq'])
    
    # Reorder rows and columns to ensure high -> medium -> low ordering
    flow_counts = flow_counts.reindex(freq_order, axis=0)
    flow_counts = flow_counts.reindex(freq_order, axis=1)
    
    flow_pct = flow_counts.div(flow_counts.sum(axis=1), axis=0) * 100
    
    # Create a custom heatmap with arrows to indicate flow
    plt.figure(figsize=(6, 3), facecolor='white')
    sns.heatmap(flow_pct, annot=True, fmt='.1f', cmap='YlGnBu', 
                linewidths=1, linecolor='white')
    
    # Add arrows to indicate migration from true class to predicted class
    for i, true_cat in enumerate(flow_pct.index):
        for j, pred_cat in enumerate(flow_pct.columns):
            if flow_pct.iloc[i, j] > 10:  # Only show significant flows
                plt.text(j + 0.5, i + 0.5, '→', 
                         ha='center', va='center', fontsize=20, 
                         color='white' if flow_pct.iloc[i, j] > 50 else 'black')
    
    plt.xlabel('Predicted Class Frequency', fontsize=14)
    plt.ylabel('True Class Frequency', fontsize=14)
    plt.title(f'Migration Pattern - {distribution_type.upper()}', fontsize=16)
    plt.tight_layout()
    
    output_path = os.path.join(output_dir, f"{prefix}{distribution_type}_class_migration_pattern.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved class migration pattern to {output_path}")
    
    return flow_counts, flow_pct

In [ ]:
def analyze_lt_distribution(root_folder, distribution_type, prefix=""):
    """Run the complete analysis for a long-tailed distribution."""
    # Get the data and preparatory information
    combined_df, freq_order, root, prefix, distribution_type = compute_misclassification_stats_by_class_frequency(
        root_folder, distribution_type, prefix)
    
    if combined_df is None:
        print("Analysis could not be completed due to data loading issues.")
        return
    
    # Create the various visualizations
    misclass_by_freq = create_misclassification_heatmap(combined_df, freq_order, root, prefix, distribution_type)
    gi_pivot, gi_by_freq = create_gi_coverage_heatmap(combined_df, freq_order, root, prefix, distribution_type)
    create_gi_bar_chart(gi_by_freq, freq_order, root, prefix, distribution_type)
    flow_counts, flow_pct = create_migration_pattern(combined_df, freq_order, root, prefix, distribution_type)
    
    # Print summary information
    print("\nMisclassification Analysis Summary:")
    print(f"Distribution type: {distribution_type}")
    print("\nAverage GI Coverage (% > 1) by Frequency Category:")
    print(gi_pivot.to_string())
    
    print("\nSample Migration Patterns (% of true class):")
    print(misclass_by_freq.to_string())
    
    print("\nMisclassification Counts by Frequency Category:")
    print(flow_counts.to_string())
    
    return {
        "misclass_heatmap": misclass_by_freq,
        "gi_coverage": gi_pivot,
        "flow_counts": flow_counts,
        "flow_percentages": flow_pct
    }

In [ ]:
# Example usage:
# LT1 Analysis
analyze_lt_distribution(
    "data/MNIST-LT1/GI_mnist_lt1_experiment_outputs", 
    "lt1", 
    "mnist_lt1_"
)

In [ ]:
# LT2 Analysis
analyze_lt_distribution(
    "data/MNIST-LT2/GI_mnist_lt2_experiment_outputs", 
    "lt2", 
    "mnist_lt2_"
)


In [ ]:
def create_combined_random_lt2_plot():
    """
    Create a combined horizontal plot with the random and LT2 plots.
    
    This combines:
    1. MNIST Random model comparison
    2. LT2 Misclassification Distribution
    3. LT2 GI Coverage
    
    All plots are arranged horizontally in a single figure.
    """
    # Define paths for the input images
    random_plot_path = os.path.join(output_dir, "MNIST-RANDOM_model_comparison_GIs.png")
    lt2_misclass_path = os.path.join(output_dir, "mnist_lt2_lt2_misclass_by_freq.png")
    lt2_coverage_path = os.path.join(output_dir, "mnist_lt2_lt2_gi_coverage_by_freq.png")
    
    # Check if all input files exist
    for path in [random_plot_path, lt2_misclass_path, lt2_coverage_path]:
        if not os.path.exists(path):
            print(f"Error: Required input file not found: {path}")
            return
    
    # Import matplotlib.image if not already imported
    import matplotlib.image as mpimg
    
    # Load the three images
    random_img = mpimg.imread(random_plot_path)
    lt2_misclass_img = mpimg.imread(lt2_misclass_path)
    lt2_coverage_img = mpimg.imread(lt2_coverage_path)
    
    # Create a horizontal figure with three subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor='white')
    
    # Display each image in its subplot
    axes[0].imshow(random_img)
    axes[0].axis('off')
    axes[0].set_title('A: MNIST Random Distribution GI', fontsize=14, y=-0.15)
    
    axes[1].imshow(lt2_misclass_img)
    axes[1].axis('off')
    axes[1].set_title('B: LT2 Misclassification Distribution', fontsize=14, y=-0.15)
    
    axes[2].imshow(lt2_coverage_img)
    axes[2].axis('off')
    axes[2].set_title('C: LT2 GI Coverage', fontsize=14, y=-0.15)
    
    # Adjust layout and save
    plt.tight_layout()
    output_path = os.path.join(output_dir, "combined_random_lt2_plot.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    
    # Also save as PDF
    output_path_pdf = os.path.join(output_dir, "combined_random_lt2_plot.pdf")
    plt.savefig(output_path_pdf, bbox_inches='tight', facecolor='white')
    
    plt.show()
    print(f"Combined plot saved to {output_path} and {output_path_pdf}")



In [ ]:
# Usage example:
create_combined_random_lt2_plot()